# AIMA Figure 17.1: value iteration and navigation

This notebook recreates the **4 × 3 grid world** in Figure 17.1 of Russell and Norvig, *Artificial Intelligence: A Modern Approach*, 4th ed. The agent starts at (1, 1), cannot enter (2, 2), and ends on +1 or −1. In stochastic mode, an action goes in the intended direction with probability 0.8 and to either perpendicular direction with probability 0.1. A wall collision leaves the agent in place.

In [21]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle, Circle
from IPython.display import HTML, display

# action costs to try: 0.04, 0.01, 0.5, 2, -0.1
action_cost = 0.04
stochastic = True       # False: every action goes exactly where intended
discount = 1.0          # Episodic, undiscounted task
seed = 7                # Makes the animated sample run repeatable
max_steps = 40          # Maximum moves in the sample run


## Bellman update

For a nonterminal state, the notebook repeatedly applies

$$V_{k+1}(s)=\max_a Q_k(s,a), \qquad
Q_k(s,a)=\sum_{s'}P(s'\mid s,a)\left[R(s,a,s')+\gamma V_k(s')\right].$$

Entering **+1** or **−1** gives that reward and ends the episode. Every other transition costs `action_cost`, including a bump into a wall. Terminal states have no future value; the plots label them with their one time payoff. All states in one iteration use the values from the *previous* iteration.

In [22]:
actions = {'↑': (0, 1), '→': (1, 0), '↓': (0, -1), '←': (-1, 0)}
directions = list(actions)
wall = (1, 1)                   # Code coordinates start at (0, 0)
terminal = {(3, 2): 1, (3, 1): -1}
states = [(x, y) for y in range(3) for x in range(4) if (x, y) != wall]

def move(state, direction):
    dx, dy = actions[direction]
    destination = (state[0] + dx, state[1] + dy)
    return destination if destination in states else state

def q_value(state, action, values):
    i = directions.index(action)
    outcomes = ([(action, 0.8), (directions[(i - 1) % 4], 0.1),
                 (directions[(i + 1) % 4], 0.1)] if stochastic else [(action, 1.0)])
    return sum(p * (terminal[next_state] if next_state in terminal
                    else -action_cost + discount * values[next_state])
               for direction, p in outcomes
               for next_state in [move(state, direction)])

values = {state: 0.0 for state in states}
history = [values.copy()]
for iteration in range(100):
    updated = {state: (0.0 if state in terminal else
                       max(q_value(state, action, values) for action in directions))
               for state in states}
    history.append(updated.copy())
    change = max(abs(updated[state] - values[state]) for state in states)
    values = updated
    if change < 1e-5:
        break

policy = {state: max(directions, key=lambda action: q_value(state, action, values))
          for state in states if state not in terminal}
print(f'Converged after {len(history) - 1} iterations (largest change: {change:.2g}).')

def draw_grid(ax, values, agent=None, trail=(), show_policy=False):
    ax.clear()
    for y in range(3):
        for x in range(4):
            state = (x, y)
            color = ('#485563' if state == wall else
                     '#c9f2d2' if state == (3, 2) else
                     '#ffd9d4' if state == (3, 1) else '#f5f8fc')
            ax.add_patch(Rectangle((x, y), 1, 1, facecolor=color,
                                   edgecolor='#546273', linewidth=1.5))
            label = ('WALL' if state == wall else
                     f'{terminal[state]:+d}' if state in terminal else
                     f'{values[state]:+.2f}')
            ax.text(x + 0.5, y + 0.52, label, ha='center', va='center', fontsize=15)
            if show_policy and state in policy:
                ax.text(x + 0.78, y + 0.78, policy[state], ha='center',
                        va='center', fontsize=15, color='#1464a1', weight='bold')
    if trail:
        ax.plot([x + 0.5 for x, y in trail], [y + 0.5 for x, y in trail],
                color='#e08e26', linewidth=3, alpha=0.8)
    if agent is not None:
        ax.add_patch(Circle((agent[0] + 0.5, agent[1] + 0.22), 0.13,
                            color='#28395b', zorder=5))
    ax.set(xlim=(0, 4), ylim=(0, 3), xticks=np.arange(0.5, 4.5),
           yticks=np.arange(0.5, 3.5), aspect='equal')
    ax.set_xticklabels([1, 2, 3, 4])
    ax.set_yticklabels([1, 2, 3])
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)


Converged after 25 iterations (largest change: 9.8e-06).


## 1. Watch the values spread through the grid

The first frame is the all zero estimate. Each following frame is one synchronous Bellman update. Green and red cells show terminal payoffs, while the other cells show the current value estimate.

In [23]:
fig, ax = plt.subplots(figsize=(8, 5))

def value_frame(i):
    draw_grid(ax, history[i])
    ax.set_title(f'Value iteration {i} of {len(history) - 1}', fontsize=16)

animation = FuncAnimation(fig, value_frame, frames=len(history), interval=350,
                          repeat=False)
display(HTML(animation.to_jshtml()))
plt.close(fig)


## 2. Follow the policy from START

At each position, compare the four **Q values** and choose the largest. Blue arrows show the learned action in every nonterminal cell. The dark circle is the agent; the orange line is its actual path. With stochastic transitions, the actual direction can differ from the chosen action. Change `seed` to watch another sample run.

In [24]:
rng = np.random.default_rng(seed)
path = [(0, 0)]
decisions = []
rewards = []

for step in range(max_steps):
    state = path[-1]
    if state in terminal:
        break
    scores = {action: q_value(state, action, values) for action in directions}
    chosen = max(scores, key=scores.get)
    i = directions.index(chosen)
    actual = (rng.choice([chosen, directions[(i - 1) % 4],
                          directions[(i + 1) % 4]], p=[0.8, 0.1, 0.1])
              if stochastic else chosen)
    next_state = move(state, actual)
    decisions.append((chosen, actual, scores))
    rewards.append(terminal[next_state] if next_state in terminal else -action_cost)
    path.append(next_state)

fig, (ax, info) = plt.subplots(1, 2, figsize=(11, 4.5),
                                gridspec_kw={'width_ratios': [4, 2.3]})

def navigation_frame(i):
    draw_grid(ax, values, agent=path[i], trail=path[:i + 1], show_policy=True)
    ax.set_title(f'Policy navigation: move {i}', fontsize=15)
    info.clear()
    info.axis('off')
    lines = [f'Current state: ({path[i][0] + 1}, {path[i][1] + 1})',
             f'Reward so far: {sum(rewards[:i]):+.2f}', '']
    if i:
        chosen, actual, _ = decisions[i - 1]
        lines += [f'Previous choice: {chosen}', f'Actual direction: {actual}', '']
    if i < len(decisions):
        chosen, _, scores = decisions[i]
        lines += ['Q(current state, action):']
        lines += [f'  {action}  {scores[action]:+.3f}' +
                  ('   ← choose' if action == chosen else '') for action in directions]
    else:
        lines += ['Terminal reached' if path[i] in terminal else
                  'Stopped at the move limit']
    info.text(0.03, 0.95, '\n'.join(lines), va='top', fontsize=13,
              family='monospace', linespacing=1.5)

animation = FuncAnimation(fig, navigation_frame, frames=len(path),
                          interval=850, repeat=False)
display(HTML(animation.to_jshtml()))
plt.close(fig)
print(f'{len(decisions)} moves; final state {tuple(v + 1 for v in path[-1])}; '
      f'total reward {sum(rewards):+.2f}')


7 moves; final state (4, 3); total reward +0.76
